# Assignment 1: EDA and Baselines

CENG313 Introduction to Data Science, BYU - Locating Bacterial Flagellar Motors 2025.

All reusable code lives in `src/`; this notebook only calls it.

## 0. Setup

Clones the repository and adds it to the Python path.

The repository is private: before running this cell, add your own GitHub token to Colab Secrets (key icon on the left) as `GITHUB_TOKEN` and enable notebook access. See README, "How to run".

In [ ]:
import os, sys
from google.colab import userdata

# Private repo: each member uses their own token from Colab Secrets.
token = userdata.get("GITHUB_TOKEN")
REPO_URL = f"https://{token}@github.com/yasunl/GROUP10_CENG313.git"
REPO_DIR = "/content/GROUP10_CENG313"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

sys.path.insert(0, REPO_DIR)
!pip -q install -r {REPO_DIR}/requirements.txt

## 1. Data download

Upload your `kaggle.json` (Kaggle -> Settings -> API -> Create New Token) when prompted.

The raw competition data is about 70 GB of JPEG slices, too large for Colab. We downsampled every training tomogram 8x on all axes on Kaggle (`kaggle/downsample_train.py`) and published the result as the private Kaggle Dataset `yasunl/group10-byu-downsampled-8x` (0.35 GB). Your Kaggle account needs access to it (see README).

After the download, `data/` contains:
- `labels_downsampled.csv`: the competition's training labels plus `z_s, y_s, x_s` (motor coordinates in the small volume = original / 8)
- `downsampled/<tomo_id>.npz`: one small volume per tomogram
- `examples/<tomo_id>.npz`: full-resolution slices around the motor for the image EDA

In [ ]:
from google.colab import files
import os

if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    uploaded = files.upload()  # choose kaggle.json
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    !mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Skip the download if a previous run in this Colab session already did it.
if not os.path.exists("data/labels_downsampled.csv"):
    !kaggle datasets download -d yasunl/group10-byu-downsampled-8x -p data --unzip
!ls data

In [ ]:
from src import data, utils, models, plots

labels = data.load_labels("data/labels_downsampled.csv")
tomos = data.tomogram_table(labels)
print(labels.shape, tomos.shape)
labels.head()

In [ ]:
# Sanity check of the downsampled data: every tomogram has a volume, and every
# motor coordinate (original / 8) falls inside its small volume.
from pathlib import Path

npz_files = sorted(Path("data/downsampled").glob("*.npz"))
print(len(npz_files), "volumes for", tomos.shape[0], "tomograms")

volume, meta = data.load_downsampled(npz_files[0])
print(npz_files[0].stem, volume.shape, volume.dtype, meta)

motors = labels[labels.has_motor]
inside = 0
for tomo_id, group in motors.groupby("tomo_id"):
    shape = data.load_downsampled(Path("data/downsampled") / f"{tomo_id}.npz")[0].shape
    inside += ((group[["z_s", "y_s", "x_s"]].to_numpy() < shape).all(axis=1)).sum()
print(f"{inside}/{len(motors)} motor coordinates inside their downsampled volume")

## 2. Data description

_TODO: n, d, instances/features, measurement level of each column (Lecture 3)._

## 3. Metadata EDA

_TODO_

In [ ]:
plots.class_balance(tomos);

## 4. Motor position EDA

_TODO_

## 5. Image EDA

_TODO_

## 6. Baselines

In [ ]:
train_ids, val_ids = utils.split_tomograms(tomos)
train_labels = labels[labels.tomo_id.isin(train_ids)]
val_labels = labels[labels.tomo_id.isin(val_ids)]
val_tomos = tomos[tomos.tomo_id.isin(val_ids)]

results = {}
for name, model in [("B0", models.NoMotorBaseline()), ("B1", models.MeanPositionBaseline())]:
    preds = model.fit(train_labels).predict(val_tomos)
    f2, counts = utils.score(val_labels, preds, return_counts=True)
    results[name] = {"F2": round(f2, 4), **counts}

import pandas as pd
pd.DataFrame(results).T

## 7. Observations

_TODO_